# Notebook 4: Vector Addition — Your First Complete CUDA Program

Vector addition is the "Hello World" of GPU computing. In this notebook, you will:

- Build a complete CUDA program from scratch
- Compare CPU vs GPU implementations
- Measure and compare performance
- Learn the grid-stride loop pattern in a real context
- Verify correctness with validation

In [ ]:
%load_ext nvcc4jupyter

## 4.1 The Problem

Given two vectors `a` and `b` of length N, compute `c[i] = a[i] + b[i]` for all `i`.

This is **embarrassingly parallel** — each element's computation is completely independent. Perfect for the GPU.

## 4.2 CPU Baseline

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>
#include <cmath>

void vector_add_cpu(const float* a, const float* b, float* c, int N) {
    for (int i = 0; i < N; i++) {
        c[i] = a[i] + b[i];
    }
}

int main() {
    const int N = 1 << 20;  // ~1 million elements
    const size_t bytes = N * sizeof(float);

    float* a = (float*)malloc(bytes);
    float* b = (float*)malloc(bytes);
    float* c = (float*)malloc(bytes);

    // Initialize with known values
    for (int i = 0; i < N; i++) {
        a[i] = sinf(i);
        b[i] = cosf(i);
    }

    // CPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    vector_add_cpu(a, b, c, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);
    printf("CPU vector add: %.3f ms\n", ms);
    printf("Verification: c[0]=%.4f (expected %.4f)\n",
           c[0], sinf(0) + cosf(0));

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    free(a); free(b); free(c);
    return 0;
}

## 4.3 GPU Version — Simple

The simplest GPU kernel: one thread per element, bounds check included.

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>
#include <cmath>

// GPU kernel: one thread per element
__global__ void vector_add(const float* a, const float* b, float* c, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        c[idx] = a[idx] + b[idx];
    }
}

int main() {
    const int N = 1 << 20;
    const size_t bytes = N * sizeof(float);

    // Host memory
    float* h_a = (float*)malloc(bytes);
    float* h_b = (float*)malloc(bytes);
    float* h_c = (float*)malloc(bytes);

    for (int i = 0; i < N; i++) {
        h_a[i] = sinf(i);
        h_b[i] = cosf(i);
    }

    // Device memory
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy inputs to device
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Launch kernel
    int block_size = 256;
    int grid_size = (N + block_size - 1) / block_size;

    // Timing (kernel only, not transfers)
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    vector_add<<<grid_size, block_size>>>(d_a, d_b, d_c, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    // Copy result back
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Verify
    int errors = 0;
    for (int i = 0; i < N; i++) {
        float expected = sinf(i) + cosf(i);
        if (fabsf(h_c[i] - expected) > 1e-5) {
            errors++;
        }
    }

    printf("GPU vector add: %.3f ms\n", ms);
    printf("Grid: %d blocks x %d threads = %d total threads\n",
           grid_size, block_size, grid_size * block_size);
    printf("Verification: %s (%d errors)\n",
           errors == 0 ? "PASSED" : "FAILED", errors);

    // Bandwidth calculation
    float gb = 3.0f * bytes / 1e9;  // Read a, b; write c
    printf("Effective bandwidth: %.2f GB/s\n", gb / (ms / 1000.0f));

    // Cleanup
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return 0;
}

## 4.4 GPU Version — Grid-Stride Loop

The production-quality version uses a grid-stride loop and can handle any N with any grid size.

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>
#include <cmath>

// Production-quality kernel with grid-stride loop
__global__ void vector_add_stride(const float* a, const float* b, float* c, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;

    for (int i = idx; i < N; i += stride) {
        c[i] = a[i] + b[i];
    }
}

int main() {
    const int N = 1 << 24;  // ~16 million elements
    const size_t bytes = N * sizeof(float);

    printf("Vector size: %d elements (%.1f MB)\n", N, bytes / 1e6);

    // Host memory
    float* h_a = (float*)malloc(bytes);
    float* h_b = (float*)malloc(bytes);
    float* h_c = (float*)malloc(bytes);

    for (int i = 0; i < N; i++) {
        h_a[i] = sinf(i);
        h_b[i] = cosf(i);
    }

    // Device memory
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Get device properties to choose good grid size
    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);
    int block_size = 256;
    // Use enough blocks to fill all SMs, but not too many
    int grid_size = prop.multiProcessorCount * 4;  // Heuristic

    printf("Launch config: %d blocks x %d threads (%d total)\n",
           grid_size, block_size, grid_size * block_size);
    printf("Elements per thread: ~%d\n", N / (grid_size * block_size));

    // Timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    vector_add_stride<<<grid_size, block_size>>>(d_a, d_b, d_c, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Verify
    int errors = 0;
    for (int i = 0; i < N; i++) {
        if (fabsf(h_c[i] - (sinf(i) + cosf(i))) > 1e-5) errors++;
    }

    printf("\nKernel time: %.3f ms\n", ms);
    printf("Verification: %s\n", errors == 0 ? "PASSED" : "FAILED");

    float gb = 3.0f * bytes / 1e9;
    printf("Effective bandwidth: %.2f GB/s\n", gb / (ms / 1000.0f));

    cudaEventDestroy(start); cudaEventDestroy(stop);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return 0;
}

## 4.5 Complete Comparison: CPU vs GPU (Including Transfer Time)

It's important to measure the **total** GPU time including data transfers, not just the kernel. For small data, the transfer overhead can make the GPU slower than the CPU!

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>
#include <cmath>

void vector_add_cpu(const float* a, const float* b, float* c, int N) {
    for (int i = 0; i < N; i++) c[i] = a[i] + b[i];
}

__global__ void vector_add_gpu(const float* a, const float* b, float* c, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;
    for (int i = idx; i < N; i += stride) {
        c[i] = a[i] + b[i];
    }
}

int main() {
    printf("%12s | %10s | %10s | %10s\n",
           "N", "CPU (ms)", "GPU+xfer", "Kernel only");
    printf("-------------|------------|------------|------------\n");

    // Test multiple sizes
    int sizes[] = {1<<10, 1<<14, 1<<18, 1<<22, 1<<24};

    for (int s = 0; s < 5; s++) {
        int N = sizes[s];
        size_t bytes = N * sizeof(float);

        float* h_a = (float*)malloc(bytes);
        float* h_b = (float*)malloc(bytes);
        float* h_c_cpu = (float*)malloc(bytes);
        float* h_c_gpu = (float*)malloc(bytes);

        for (int i = 0; i < N; i++) {
            h_a[i] = (float)i;
            h_b[i] = (float)(N - i);
        }

        float *d_a, *d_b, *d_c;
        cudaMalloc(&d_a, bytes);
        cudaMalloc(&d_b, bytes);
        cudaMalloc(&d_c, bytes);

        cudaEvent_t t0, t1, t2, t3;
        cudaEventCreate(&t0);
        cudaEventCreate(&t1);
        cudaEventCreate(&t2);
        cudaEventCreate(&t3);

        // CPU timing
        cudaEventRecord(t0);
        vector_add_cpu(h_a, h_b, h_c_cpu, N);
        cudaEventRecord(t1);
        cudaEventSynchronize(t1);
        float cpu_ms;
        cudaEventElapsedTime(&cpu_ms, t0, t1);

        // GPU timing (total including transfers)
        int block_size = 256;
        int grid_size = (N + block_size - 1) / block_size;

        cudaEventRecord(t0);
        cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
        cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);
        cudaEventRecord(t1);  // After transfer
        vector_add_gpu<<<grid_size, block_size>>>(d_a, d_b, d_c, N);
        cudaEventRecord(t2);  // After kernel
        cudaMemcpy(h_c_gpu, d_c, bytes, cudaMemcpyDeviceToHost);
        cudaEventRecord(t3);  // After everything
        cudaEventSynchronize(t3);

        float gpu_total, kernel_ms;
        cudaEventElapsedTime(&gpu_total, t0, t3);
        cudaEventElapsedTime(&kernel_ms, t1, t2);

        printf("%12d | %10.3f | %10.3f | %10.3f\n",
               N, cpu_ms, gpu_total, kernel_ms);

        cudaEventDestroy(t0); cudaEventDestroy(t1);
        cudaEventDestroy(t2); cudaEventDestroy(t3);
        cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
        free(h_a); free(h_b); free(h_c_cpu); free(h_c_gpu);
    }

    printf("\nNote: GPU wins only when N is large enough to amortize transfer cost.\n");
    return 0;
}

### Key observations:

1. **Small N (1K–16K):** CPU is faster! Transfer overhead dominates.
2. **Medium N (~256K):** Roughly equal — the crossover point.
3. **Large N (1M+):** GPU wins, especially for the kernel alone.
4. **Transfer cost** is the dominant factor for memory-bound operations like vector add.

Vector addition does very little compute per byte transferred (1 add per 12 bytes read/written). It's **memory-bound**. More compute-heavy kernels benefit more from the GPU.

## 4.6 Scaling Experiment: Effect of Block Size

Let's see how block size affects performance.

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>

__global__ void vector_add(const float* a, const float* b, float* c, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) c[idx] = a[idx] + b[idx];
}

int main() {
    const int N = 1 << 22;  // 4M elements
    const size_t bytes = N * sizeof(float);

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Initialize
    float* h_a = (float*)malloc(bytes);
    float* h_b = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) { h_a[i] = 1.0f; h_b[i] = 2.0f; }
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    printf("%10s | %10s | %10s\n", "Block size", "Grid size", "Time (ms)");
    printf("-----------|------------|----------\n");

    int block_sizes[] = {32, 64, 128, 256, 512, 1024};

    for (int b = 0; b < 6; b++) {
        int bs = block_sizes[b];
        int gs = (N + bs - 1) / bs;

        cudaEvent_t start, stop;
        cudaEventCreate(&start);
        cudaEventCreate(&stop);

        // Warmup
        vector_add<<<gs, bs>>>(d_a, d_b, d_c, N);
        cudaDeviceSynchronize();

        // Timed run
        cudaEventRecord(start);
        vector_add<<<gs, bs>>>(d_a, d_b, d_c, N);
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float ms;
        cudaEventElapsedTime(&ms, start, stop);
        printf("%10d | %10d | %10.3f\n", bs, gs, ms);

        cudaEventDestroy(start);
        cudaEventDestroy(stop);
    }

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b);
    return 0;
}

**Typical result:** Block sizes 128–512 perform similarly for simple kernels. Very small (32) or very large (1024) can be slightly worse. **256 is a safe default.**

## 4.7 Key Takeaways

1. **Vector addition** is the canonical first CUDA program — simple but teaches all fundamentals
2. **Always verify correctness** — compare GPU output against CPU reference
3. **Include transfer time** in benchmarks — kernel-only timing is misleading
4. **GPU isn't always faster** — for small data or low compute intensity, CPU wins
5. **Memory-bound kernels** (like vector add) are limited by bandwidth, not compute
6. **Block size 256** is a good default

## Exercises

**Exercise 4.1:** Implement vector subtraction, multiplication, and division as separate kernels. Time each one. Are they all the same speed? Why or why not?

**Exercise 4.2:** Implement SAXPY: `y[i] = a * x[i] + y[i]` where `a` is a scalar. This is a key BLAS operation. Compare with CPU.

**Exercise 4.3:** Find the crossover point on your GPU: what's the smallest N where the GPU (including transfers) beats the CPU for vector addition?

In [ ]:
%%cuda
// Exercise 4.2 — SAXPY
#include <cstdio>
#include <cstdlib>
#include <cmath>

__global__ void saxpy(float a, const float* x, float* y, int N) {
    // TODO: y[i] = a * x[i] + y[i]
}

int main() {
    const int N = 1 << 20;
    const size_t bytes = N * sizeof(float);
    const float a = 2.5f;

    // TODO: Allocate, initialize, launch, verify
    // Hint: x = all 1.0, y = all 2.0 → result should be 2.5*1.0 + 2.0 = 4.5

    return 0;
}

---
**Next:** [Notebook 5 — Error Handling](05_Error_Handling.ipynb) — Making your CUDA programs robust and debuggable.